In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col, trim, StringType
from itertools import chain
from pyspark.sql import Window

#Reading From Silver Table

In [0]:
df = spark.table("workspace.silver.transtractions")

#Read gold dimention tables

In [0]:
df_promo_cal = spark.table("workspace.gold.dim_promo_calander")
df_store = spark.table("workspace.gold.dim_store")

#Transform

##Create primary key

###Promo Key

In [0]:
df_promo_cal.display()

In [0]:
print("rows:", df.count())
print("columns:", len(df.columns))

In [0]:
df.display()

In [0]:
df_promo_cal = df_promo_cal.select(
    F.col("store_id").alias("d_store_id"),
    F.col("category").alias("d_category"),
    F.col("start_date").alias("d_start"),
    F.col("end_date").alias("d_end"),
    "promo_key",

)
cond = (
    (df_promo_cal['d_store_id']==df['store_id']) &
    (df_promo_cal['d_category']==df['category']) &
    (df_promo_cal['d_start']<=df['order_date']) &
    (df_promo_cal['d_end']>=df['order_date'])
)

df = (df.join(df_promo_cal, on = cond, how = 'left')
      .drop('promo_id','promo_name','d_store_id','d_category','d_start','d_end'))

df = df.select("promo_key", *[c for c in df.columns if c != "promo_key"])

In [0]:
df = df.fillna({"promo_key": 0})

In [0]:
print("rows:", df.count())
print("columns:", len(df.columns))

In [0]:
df.display()

###Sales Key

In [0]:
df_store.display()

In [0]:
df_store = df_store.select(
    F.col("store_id").alias("d_store_id"),
    "store_key",
)

cond = (
    (df_store['d_store_id']==df['store_id'])
)

df = (df.join(df_store, on = cond, how = 'left')
      .drop('d_store_id'))

df = df.select("store_key", *[c for c in df.columns if c != "store_key"])


In [0]:
print("rows:", df.count())
print("columns:", len(df.columns))

In [0]:
df.display()

In [0]:
df.filter(F.col('store_key').isNull()).display()

#Write into gold layer

In [0]:
(
    df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold.fact_tanstractions")
)